In [ ]:
import sys, os
from pathlib import Path

# Add src to sys.path for modular imports
root_dir = Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import json, importlib.util, zipfile, joblib, math, random, numbers
import itertools
from dataclasses import dataclass
from typing import List, Sequence, Tuple, Union, Optional, Callable, Dict, Any, Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch import nn, Tensor
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
from xgboost import XGBClassifier

# Modular imports from src
from src.utils.general_utils import set_deterministic, periods_per_year, in_kaggle, in_colab
from src.utils.data_utils import split_scale, triple_barrier_label, CryptoDataset, data_pipe, make_utility_class_weights, fetch_data
from src.utils.metrics import f1_metric, brier_metric, bss_metric, triple_barrier_metrics
from src.models.DL_models import LSTMClassifier, BiLSTMClassifier, GRUClassifier, BiGRUClassifier, LSTMCatBoostClassifier, LSTMLightGBMClassifier, LSTMXGBClassifier, GRUCatBoostClassifier, GRULightGBMClassifier, GRUXGBClassifier
from src.utils.trading_utils import TradingStrategy, TripleBarrierConfig, standard_trade_test, sweep_min_return


In [ ]:
# Globals
tfs = ['1h', '4h']
es_names = ['f1', 'bss', 'brier']
KU_KD_GRID = [(2, 1), (3, 1), (4, 2)]
GRU_CONFIGS = [{'hidden_size': 128, 'num_layers': 2, 'dropout': 0.1}]
from src.utils.general_utils import walk
def _ensure_dir(path): 
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p

## Trade utils

In [31]:
def buy_and_hold_stats(
    prices: pd.DataFrame,
    *,
    equity: float = 10000.0,
    price_col: str = "close",
) -> dict:
    """
    Compute simple Buy&Hold stats on a price series:

    - total_return: % change from first to last price
    - final_equity: equity * (last_price / first_price)
    - max_drawdown: min equity / rolling max - 1 (negative fraction)
    """

    # --- extract and clean price series ---
    if price_col not in prices.columns:
        raise ValueError(f"prices must contain '{price_col}' column")

    px = prices[price_col].astype(float).dropna()
    if px.shape[0] < 2:
        return {
            "total_return": 0.0,
            "final_equity": float(equity),
            "max_drawdown": 0.0,
        }

    # --- equity curve under BnH ---
    # scale prices so that first point corresponds to `equity`
    eq = equity * (px / px.iloc[0])

    # total return in %
    total_return = (eq.iloc[-1] / eq.iloc[0] - 1.0) * 100.0
    final_equity = float(eq.iloc[-1])

    # max drawdown as in calculate_metrics (negative fraction)
    run_max = np.maximum.accumulate(eq.values)
    dd = eq.values / run_max - 1.0
    max_drawdown = float(dd.min())  # e.g. -0.35 for -35%

    return {
        "total_return": total_return,
        "final_equity": final_equity,
        "max_drawdown": max_drawdown,
    }

## Definitions

In [32]:
MODELS = {
        "lstm": LSTMClassifier,
        "bilstm": BiLSTMClassifier,
        "gru": GRUClassifier,
        "bigru": BiGRUClassifier,
        "lstm_cat": LSTMCatBoostClassifier,
        "lstm_lgb": LSTMLightGBMClassifier,
        "lstm_xgb": LSTMXGBClassifier,
        "gru_cat": GRUCatBoostClassifier,
        'gru_xgb': GRUXGBClassifier,
        'gru_lgb': GRULightGBMClassifier,
}

In [33]:
# FEATURE BLOCKS 

ALL_FEATURES = [
    'volume', 'funding_rate',
    'ema_20', 'ema_50', 'ema_200', 'adx', 'rsi_14',
    'bb_percent', 'bb_width',
    'atr_vol_regime',

    'mfi_14', 'z_volume',
    'vwap', 'vwap_distance',

    'log_return_1h', 'log_return_4h', 'log_return_1d',
    'roll_std_4h', 'roll_std_8h',

    'funding_bias',
    'vwap_session', 'session_vol_mean', 'session_return_mean', 'session_volatility',

    # FD / fractal block
    'fd_24', 'fd_slope', 'fd_ema_12', 'fd_ema_24',
    'fd_trend_strength', 'fd_threshold_causal', 'fd_regime', 'fd_regime_switch',
    'fd_volatility', 'fd_vol_ratio', 'fd_vol_slope', 'fd_slope_atr_norm',
    'fd_entropy', 'fd_vol_adjusted',
    'fd_24_robust_z', 'fd_ema_12_robust_z', 'fd_ema_24_robust_z',
    'fd_trend_strength_robust_z', 'fd_slope_robust_z',
    'fd_slope_atr_norm_robust_z', 'fd_volatility_robust_z',
    'fd_vol_ratio_robust_z', 'fd_vol_slope_robust_z',
    'fd_entropy_robust_z', 'fd_vol_adjusted_robust_z',

    # Patterns / ICT
    'pattern_bullish_engulf', 'pattern_bearish_engulf',
    'pattern_harami', 'pattern_hammer', 'pattern_inverted_hammer',
    'swing_high', 'swing_low', 'last_swing_high', 'last_swing_low',
    'bos_bullish', 'bos_bearish',
    'choch_bullish', 'choch_bearish',
    'mss_bullish', 'mss_bearish',
    'bullish_fvg', 'bearish_fvg', 'fvg_gap',
    'rolling_high', 'rolling_low', 'equilibrium',
    'breakout_bullish', 'breakout_bearish',
    'pattern_count', 'pattern_active',

    # Macro / on-chain
    'macro_event_sentiment', 'macro_event_flag',
    'macro_event_intensity', 'macro_event_intensity_smooth',
    'fear_greed', 'onchain_activity_index',

    # ATR / range & levels
    'atr_pct', 'range_atr', 'body_atr',
    'dist_daily_high', 'dist_daily_low',
    'dist_weekly_high', 'dist_weekly_low',
    'atr_vol_regime_z',

    # Extra TA
    'ppo', 'ppo_signal', 'ppo_hist',
    'bb_z', 'bb_percB',

    # Time / sessions / PDA
    'hour_sin', 'hour_cos',
    'dayofweek_sin', 'dayofweek_cos',
    'session_Asia', 'session_Frankfurt',
    'session_London', 'session_NewYork', 'session_OffHours',
    'pda_Discount', 'pda_Premium',
]

FEATURE_BLOCKS = {
    # core price/vol + basic trend/momentum
    "core_trend_mom": [
        'volume', 'funding_rate',
        'ema_20', 'ema_50', 'ema_200', 'adx', 'rsi_14',
        'bb_percent', 'bb_width',
        'log_return_1h', 'log_return_4h', 'log_return_1d',
        'roll_std_4h', 'roll_std_8h',
        'atr_vol_regime',
        'ppo', 'ppo_signal', 'ppo_hist',
        'bb_z', 'bb_percB',
    ],

    # volume / VWAP / intraday session stats
    "volume_vwap_session": [
        'mfi_14', 'z_volume',
        'vwap', 'vwap_distance',
        'vwap_session', 'session_vol_mean',
        'session_return_mean', 'session_volatility',
    ],

    # fractal / FD regime block
    "fractal_regime": [
        'fd_24', 'fd_slope', 'fd_ema_12', 'fd_ema_24',
        'fd_trend_strength', 'fd_threshold_causal',
        'fd_regime', 'fd_regime_switch',
        'fd_volatility', 'fd_vol_ratio', 'fd_vol_slope',
        'fd_slope_atr_norm', 'fd_entropy', 'fd_vol_adjusted',
        'fd_24_robust_z', 'fd_ema_12_robust_z', 'fd_ema_24_robust_z',
        'fd_trend_strength_robust_z', 'fd_slope_robust_z',
        'fd_slope_atr_norm_robust_z', 'fd_volatility_robust_z',
        'fd_vol_ratio_robust_z', 'fd_vol_slope_robust_z',
        'fd_entropy_robust_z', 'fd_vol_adjusted_robust_z',
    ],

    # ICT-style patterns / structure / FVG / breakouts
    "patterns_ict": [
        'pattern_bullish_engulf', 'pattern_bearish_engulf',
        'pattern_harami', 'pattern_hammer', 'pattern_inverted_hammer',
        'swing_high', 'swing_low', 'last_swing_high', 'last_swing_low',
        'bos_bullish', 'bos_bearish',
        'choch_bullish', 'choch_bearish',
        'mss_bullish', 'mss_bearish',
        'bullish_fvg', 'bearish_fvg', 'fvg_gap',
        'rolling_high', 'rolling_low', 'equilibrium',
        'breakout_bullish', 'breakout_bearish',
        'pattern_count', 'pattern_active',
    ],

    # macro / on-chain
    "macro_onchain": [
        'macro_event_sentiment', 'macro_event_flag',
        'macro_event_intensity', 'macro_event_intensity_smooth',
        'fear_greed', 'onchain_activity_index',
        'funding_bias',
    ],

    # ATR / range & higher-timeframe levels
    "atr_levels": [
        'atr_pct', 'range_atr', 'body_atr',
        'dist_daily_high', 'dist_daily_low',
        'dist_weekly_high', 'dist_weekly_low',
        'atr_vol_regime_z',
    ],

    # time-of-day, weekday, session, PDA
    "time_pda": [
        'hour_sin', 'hour_cos',
        'dayofweek_sin', 'dayofweek_cos',
        'session_Asia', 'session_Frankfurt',
        'session_London', 'session_NewYork', 'session_OffHours',
        'pda_Discount', 'pda_Premium',
    ],
}



In [34]:
essential_features = [
    # Core price/vol/funding
    "volume",
    "funding_rate",
    "mfi_14",
    "z_volume",
    "log_return_15m",
    "log_return_1h",
    "log_return_4h",
    "log_return_1d",
    "roll_std_16",
    "roll_std_32",
    "funding_bias",
    "atr_vol_regime",
    "atr_pct",
    "range_atr",
    "body_atr",

    # Trend / momentum / bands
    "ema_20",
    "ema_50",
    "ema_200",
    "adx",
    "rsi_14",
    "bb_percent",
    "bb_width",
    "bb_percB",
    "ppo",
    "ppo_signal",
    "ppo_hist",

    # Fractal subset
    "fd_96",
    "fd_slope",
    "fd_trend_strength",
    "fd_volatility",
    "fd_vol_ratio",

    # VWAP & intraday stats
    "vwap",
    "vwap_distance",
    "vwap_session",
    "session_vol_mean",
    "session_return_mean",
    "session_volatility",

    # Time-of-day / sessions
    "hour_sin",
    "hour_cos",
    "dayofweek_sin",
    "dayofweek_cos",
    "session_Asia",
    "session_London",
    "session_NewYork",
    "session_OffHours",

    # Macro / on-chain / basis
    "macro_event_flag",
    "macro_event_intensity_smooth",
    "macro_event_sentiment",
    "fear_greed",
    "onchain_activity_index",
    "pda_Discount",
    "pda_Premium",
]


In [35]:
es_metrics = {
    'f1': (f1_metric, 'max'),
    'bss': (bss_metric, 'max'),
    'brier': (brier_metric, 'min'),
}

## Experiment pipes

In [ ]:
def train_predict(
    train_data, 
    val_data, 
    lr, 
    base_name, 
    input_size, 
    model_type, 
    base_dir='artifacts', 
    dropout = 0.3, 
    num_layers = 2, 
    hidden_size = 128, 
    weights=None, 
    ku=2, 
    kd=1, 
    probability_col=['p2'],
    es_metric = None
):

    if es_metric is not None:
        early_metric_fn = es_metric[0]
        early_metric_mode = es_metric[1]
    else:
        early_metric_fn = None
        early_metric_mode = None
    
    base_dir = f"{base_dir}/{model_type}_{base_name}"
    if model_type in ['gru_cat', 'gru_xgb', 'gru_lgb', 'lstm_cat', 'lstm_xgb', 'lstm_lgb']:
        model_path = base_dir
        loss_plot_path = f"{base_dir}/{model_type}_{base_name}.png"
    else:
        model_path = f"{base_dir}/{model_type}_{base_name}.pt"
        loss_plot_path = f"{base_dir}/{model_type}_{base_name}.png"
    model = MODELS[model_type](
        input_size=input_size, 
        dropout=dropout, 
        num_layers=num_layers, 
        hidden_size=hidden_size, 
        num_classes=num_classes)
    
    output = model.train(train_data, 
                         val_data, 
                         model_path=model_path, 
                         loss_plot_path=loss_plot_path, 
                         batch_size=batch_size, lr=lr, 
                         weights=weights,
                         early_metric_fn=early_metric_fn,
                         early_metric_mode=early_metric_mode,
                        )
    prediction = model.predict(val_data, batch_size=batch_size*2)
    prediction_train = model.predict(train_data, batch_size=batch_size*2)


    train_metrics = triple_barrier_metrics(
                            y_true=prediction_train["true"],
                            y_pred=prediction_train["pred"],
                            p_all=prediction_train[probability_col],
                            ku=ku,
                            kd=kd,
                        )
    val_metrics = triple_barrier_metrics(
                            y_true=prediction["true"],
                            y_pred=prediction["pred"],
                            p_all=prediction[probability_col],
                            ku=ku,
                            kd=kd,
                        )

    output['val_metrics'] = val_metrics
    output['train_metrics'] = train_metrics
    

    res_json = walk(output)
    with open(_ensure_dir(f"{base_dir}/{model_type}_{base_name}/res-{model_type}_{base_name}.json"), "w", encoding="utf-8") as f:
        json.dump(res_json, f, ensure_ascii=False, indent=2)

    return output

In [ ]:
def train_DL_panel(df_train,
                   df_val,
                   ku, 
                   kd, 
                   hold, 
                   window_size, 
                   lr, 
                   base_name, 
                   base_dir = 'artifacts', 
                   target = ['y'], 
                   model_types=['lstm', 'bilstm', 'gru'], 
                #    min_return = 1.0,
                   hidden_size = 128,
                   num_layers = 2,
                   dropout=0.3,
                   probability_col=['p2'],
                   volatility_col='atr_200',
                   es_metric=None,
                  ):
    
    res = {}
    
    base_name = f'{base_name}_ku{ku}_kd{kd}_hold{hold}_base-window{window_size}_dropout{dropout}_hidden{hidden_size}_layers{num_layers}'
    
    input_size = df_train.drop(
        columns=['open', 'high', 'low', 'close'] + [volatility_col] + target).shape[1]

    print(df_train[target].value_counts(normalize=True))
    train_data = CryptoDataset(
        df_train.drop(columns=['open', 'high', 'low', 'close'] + [volatility_col]), window_size=window_size, target=target)
    val_data = CryptoDataset(df_val.drop(columns=['open', 'high', 'low', 'close'] + [volatility_col]), window_size=window_size, target=target)

    y_train = df_train[target[0]].to_numpy().astype(int)
    cw = make_utility_class_weights(y_train, ku=ku, kd=kd, mode="balanced")
    
    print(cw)

    for model_type in model_types:
        print('Training ', model_type)        
        res[model_type] = train_predict(train_data, 
                                        val_data, 
                                        lr, 
                                        base_name, 
                                        input_size, 
                                        model_type, 
                                        base_dir=base_dir, 
                                        dropout = dropout, 
                                        num_layers = num_layers, 
                                        hidden_size = hidden_size,
                                        weights=cw,
                                        ku=ku,
                                        kd=kd,
                                        probability_col=probability_col,
                                        es_metric=es_metric
                                        )    
    
    return res

In [38]:
def load_predict_DL(
    df,
    prices,
    ku, 
    kd, 
    hold, 
    window_size, 
    base_name, 
    base_dir='artifacts', 
    dropout=0.1,
    hidden_size=128,
    num_layers=2,
    target=['y'],
    probability_col=['p2'],
    model_type='gru', 
    min_return=[0],
    slippage=0.00,
    transaction_cost=True,
    position_size=0.01,
    use_limit=False,
    limit_offset=0.0,
    risk_mode='fixed_risk',
    *,
    mr_for_mask: float = 0.0,
    volatility_col='atr_14',
    tf = '15m',
):
    """
    Load DL model, predict, run trading for given min_return values (no calibration),
    and compute model metrics (raw + masked).

    Returns:
      res[mr]:
        'trade_metrics', 'trade_log', 'equity_curve'
      res['predictions']
      res['model_metrics_raw']
      res['model_metrics_masked']
      res['mask']
    """

    res = {}
        
    base_name = (
        f'{base_name}_ku{ku}_kd{kd}_hold{hold}_base-window{window_size}_'
        f'dropout{dropout}_hidden{hidden_size}_layers{num_layers}'
    )
    
    base_dir = f"{base_dir}/{model_type}_{base_name}"
    if model_type in ['gru_cat', 'gru_xgb', 'gru_lgb', 'lstm_cat', 'lstm_xgb', 'lstm_lgb']:
        model_path = base_dir
    else:
        model_path = f"{base_dir}/{model_type}_{base_name}.pt"
    
    input_size = df.drop(
        columns=['open', 'high', 'low', 'close']+ [volatility_col] + target
    ).shape[1]

    data = CryptoDataset(
        df.drop(columns=['open', 'high', 'low', 'close']+ [volatility_col]),
        window_size=window_size,
        target=target
    )
    
    print(df[target].value_counts(normalize=True))

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
   
    if model_type in ['gru_cat', 'gru_xgb', 'gru_lgb', 'lstm_cat', 'lstm_xgb', 'lstm_lgb']:
        model = MODELS[model_type].load(model_path)
    else:
        model = MODELS[model_type](
            input_size=input_size,
            dropout=dropout,
            hidden_size=hidden_size,
            num_layers=num_layers,
            num_classes=num_classes
        )
        state_dict = torch.load(model_path, map_location=device)
        model._load_state_dict(state_dict)
    
    predictions = model.predict(data, batch_size=batch_size*2)

    # --- Build decision mask using reference mr_for_mask ---
    metrics_mask, tlog_mask, equity_info_mask = standard_trade_test(
        predictions=predictions,
        prices=prices,
        ku=ku,
        kd=kd,
        hold=hold,
        probability_column=probability_col,
        atr_column=volatility_col,
        equity=10000.0,
        position_size=position_size,
        risk_mode=risk_mode,
        compound=True,
        transaction_cost=transaction_cost,
        slippage=slippage,
        min_return=mr_for_mask,
        use_limit=use_limit,
        limit_offset=limit_offset,
        tf=tf
    )

    positions = np.asarray(equity_info_mask["positions"], dtype=int)
    if positions.shape[0] != len(predictions):
        raise ValueError("positions length does not match predictions length")
    mask = (positions == 0)

    # --- Model metrics (raw, masked) ---
    model_metrics_raw = triple_barrier_metrics(
        y_true=predictions["true"],
        y_pred=predictions["pred"],
        p_all=predictions[probability_col],
        ku=ku,
        kd=kd,
    )

    if mask.any():
        df_masked = predictions.loc[mask].copy()
        model_metrics_masked = triple_barrier_metrics(
            y_true=df_masked["true"],
            y_pred=df_masked["pred"],
            p_all=df_masked[probability_col],
            ku=ku,
            kd=kd,
        )
        model_metrics_masked["num_decision_bars"] = int(mask.sum())
    else:
        model_metrics_masked = {
            "error": "no decision bars",
            "num_decision_bars": 0,
        }

    # --- Trading for each mr ---
    for mr in min_return:
        tm, tlog, eq_info = standard_trade_test(
            predictions=predictions,
            prices=prices,
            ku=ku,
            kd=kd,
            hold=hold,
            probability_column=probability_col,
            atr_column=volatility_col,
            equity=10000.0,
            position_size=position_size,
            risk_mode="fixed_risk",
            compound=True,
            transaction_cost=transaction_cost,
            slippage=slippage,
            min_return=mr,
            use_limit=use_limit,
            limit_offset=limit_offset,
            tf=tf
        )
        res[mr] = {
            "trade_metrics": tm,
            "trade_log": tlog,
            "equity_curve": eq_info,
        }

    res["predictions"] = predictions
    res["model_metrics_raw"] = model_metrics_raw
    res["model_metrics_masked"] = model_metrics_masked
    res["mask"] = mask

    return res


In [41]:
def make_prices(df_split, volatility_col, window_size):
    return (
        df_split[["high", "low", "close", "y", volatility_col]]
        .iloc[window_size - 1 :]
        .reset_index(drop=True)
    )

def fit_logreg_baseline(df_train, window_size, volatility_col, target_col="y"):
    feature_cols = [
        c
        for c in df_train.columns
        if c not in ["open", "high", "low", "close", volatility_col, target_col]
    ]
    df_win = df_train.iloc[window_size - 1 :]
    X_train = df_win[feature_cols].values
    y_train = df_win[target_col].values

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, y_train)
    return clf, feature_cols

def make_pred_df_from_logreg(
    clf,
    df_split,
    feature_cols,
    window_size,
    probability_col,
):
    prob_col_name = probability_col if isinstance(probability_col, str) else probability_col[0]

    df_win = df_split.iloc[window_size - 1 :].reset_index(drop=True)
    X = df_win[feature_cols].values
    y_true = df_win["y"].values
    proba = clf.predict_proba(X)[:, 1]

    pred_df = pd.DataFrame(
        {
            "true": y_true,
            "pred": (proba >= 0.5).astype(int),
            prob_col_name: proba,
        }
    )
    return pred_df

def evaluate_predictions_generic(
    predictions: pd.DataFrame,
    prices: pd.DataFrame,
    min_returns,
    *,
    tf: str,
    ku: float,
    kd: float,
    hold: int,
    volatility_col: str,
    probability_col,
    position_size: float,
    risk_mode: str,
    use_limit: bool,
    limit_offset: float,
    tc: bool,
    mr_for_mask: float = 0.0,
):
    """
    Generic evaluation (classification + trading) given predictions and prices.
    Returns a dict in the same style as load_predict_DL.
    """
    res = {}
    prob_col_name = probability_col if isinstance(probability_col, str) else probability_col[0]

    # --- mask via triple-barrier backtest at mr_for_mask ---
    metrics_mask, tlog_mask, eq_info_mask = standard_trade_test(
        predictions=predictions,
        prices=prices,
        ku=ku,
        kd=kd,
        hold=hold,
        probability_column=prob_col_name,
        atr_column=volatility_col,
        equity=10000.0,
        position_size=position_size,
        risk_mode=risk_mode,
        compound=True,
        transaction_cost=tc,
        slippage=0.0,
        min_return=mr_for_mask,
        use_limit=use_limit,
        limit_offset=limit_offset,
        tf=tf,
    )

    positions = np.asarray(eq_info_mask["positions"], dtype=int)
    if positions.shape[0] != len(predictions):
        raise ValueError("positions length does not match predictions length")
    mask = positions == 0

    # --- model metrics (raw + masked) ---
    p_all = (
        predictions[[prob_col_name]]
        if isinstance(probability_col, str)
        else predictions[probability_col]
    )
    model_metrics_raw = triple_barrier_metrics(
        y_true=predictions["true"],
        y_pred=predictions["pred"],
        p_all=p_all,
        ku=ku,
        kd=kd,
    )

    if mask.any():
        df_masked = predictions.loc[mask].copy()
        p_all_masked = (
            df_masked[[prob_col_name]]
            if isinstance(probability_col, str)
            else df_masked[probability_col]
        )
        model_metrics_masked = triple_barrier_metrics(
            y_true=df_masked["true"],
            y_pred=df_masked["pred"],
            p_all=p_all_masked,
            ku=ku,
            kd=kd,
        )
        model_metrics_masked["num_decision_bars"] = int(mask.sum())
    else:
        model_metrics_masked = {
            "error": "no decision bars",
            "num_decision_bars": 0,
        }

    # --- trading metrics for each mr ---
    for mr in min_returns:
        metrics, tlog, eq_info = standard_trade_test(
            predictions=predictions,
            prices=prices,
            ku=ku,
            kd=kd,
            hold=hold,
            probability_column=prob_col_name,
            atr_column=volatility_col,
            equity=10000.0,
            position_size=position_size,
            risk_mode=risk_mode,
            compound=True,
            transaction_cost=tc,
            slippage=0.0,
            min_return=mr,
            use_limit=use_limit,
            limit_offset=limit_offset,
            tf=tf,
        )
        res[mr] = {
            "trade_metrics": metrics,
            "trade_log": tlog,
            "equity_curve": eq_info,
        }

    res["predictions"] = predictions
    res["model_metrics_raw"] = model_metrics_raw
    res["model_metrics_masked"] = model_metrics_masked
    res["mask"] = mask

    return res

# ---------------------------------------------------------
# MAIN PIPELINE
# dfs_by_tf = {"1h": df_1h, "4h": df_4h}
# ---------------------------------------------------------
def run_full_comparison(dfs_by_tf):
    """
    dfs_by_tf: dict {"1h": df_1h, "4h": df_4h}
    Returns: all_results[(tf, es_name)] = dict with DL models + ML baseline.
    """
    all_results = {}

    for tf in tfs:
        df = dfs_by_tf[tf].copy()

        # --- tf-specific params ---
        if tf == "1h":
            min_trades = 115
            hold = 332
            window_size = 332
        elif tf == "4h":
            min_trades = 0
            hold = 84
            window_size = 84
        else:
            raise ValueError(f"Unknown tf: {tf}")

        # --- split & prices ---
        df_train, df_val, df_test, scaler = data_pipe(
            df, ku, kd, hold, window_size, volatility_col=volatility_col
        )

        val_prices = make_prices(df_val, volatility_col, window_size)
        test_prices = make_prices(df_test, volatility_col, window_size)

        val_bnh = (val_prices["close"].iloc[-1] / val_prices["close"].iloc[0] - 1) * 100
        test_bnh = (test_prices["close"].iloc[-1] / test_prices["close"].iloc[0] - 1) * 100

        # -------------------------------------------------
        # ML BASELINE (logreg) – trained once per tf
        # -------------------------------------------------
        logreg, feature_cols = fit_logreg_baseline(df_train, window_size, volatility_col)

        prob_col_name = (
            probability_col if isinstance(probability_col, str) else probability_col[0]
        )

        baseline_val_pred = make_pred_df_from_logreg(
            logreg,
            df_val,
            feature_cols,
            window_size,
            probability_col,
        )
        baseline_test_pred = make_pred_df_from_logreg(
            logreg,
            df_test,
            feature_cols,
            window_size,
            probability_col,
        )

        # sweep min_return for baseline on VAL
        baseline_best = sweep_min_return(
            prices=val_prices,
            df_pred=baseline_val_pred,
            ku=ku,
            kd=kd,
            hold=hold,
            min_grid=min_returns,
            artifacts_dir=DATA_DIR,
            slippage=0.0,
            transaction_cost=tc,
            compound=True,
            use_limit=use_limit,
            limit_offset=limit_offset,
            min_trades=min_trades,
            volatility_col=volatility_col,
            probability_col=prob_col_name,
            tf=tf,
            risk_mode=risk_mode,
            position_size=position_size,
        )

        mrs_baseline = [
            baseline_best["best_return"]["mr"],
            baseline_best["best_martin"]["mr"],
            *base_mrs,
        ]
        # unique, non-None
        mrs_baseline = [mr for mr in dict.fromkeys(mrs_baseline) if mr is not None]

        baseline_val = evaluate_predictions_generic(
            baseline_val_pred,
            val_prices,
            mrs_baseline,
            tf=tf,
            ku=ku,
            kd=kd,
            hold=hold,
            volatility_col=volatility_col,
            probability_col=probability_col,
            position_size=position_size,
            risk_mode=risk_mode,
            use_limit=use_limit,
            limit_offset=limit_offset,
            tc=tc,
            mr_for_mask=0.0,
        )
        baseline_test = evaluate_predictions_generic(
            baseline_test_pred,
            test_prices,
            mrs_baseline,
            tf=tf,
            ku=ku,
            kd=kd,
            hold=hold,
            volatility_col=volatility_col,
            probability_col=probability_col,
            position_size=position_size,
            risk_mode=risk_mode,
            use_limit=use_limit,
            limit_offset=limit_offset,
            tc=tc,
            mr_for_mask=0.0,
        )

        # -------------------------------------------------
        # DL MODELS for each ES metric
        # -------------------------------------------------
        for metric_name in es_names:
            base_name = f"rf-vtc_{tf}_new-onchain_{metric_name}"

            # train panel (GRU/LSTM) for this tf + ES metric
            _ = train_DL_panel(
                df_train,
                df_val,
                ku,
                kd,
                hold,
                window_size,
                lr,
                base_name,
                base_dir=DATA_DIR,
                target=target,
                model_types=model_types,
                hidden_size=hidden_size,
                num_layers=num_layers,
                dropout=dropout,
                volatility_col=volatility_col,
                probability_col=probability_col,
                es_metric=es_metrics[metric_name],
            )

            # VAL: predictions + sweep
            val_results = {}
            best_per_model = {}
            best_mrs_per_model = {}

            for model_type in model_types:
                val_res = load_predict_DL(
                    df_val,
                    val_prices,
                    ku,
                    kd,
                    hold,
                    window_size,
                    base_name,
                    base_dir=DATA_DIR,
                    dropout=dropout,
                    hidden_size=hidden_size,
                    num_layers=num_layers,
                    target=target,
                    model_type=model_type,
                    min_return=base_mrs,
                    slippage=0.0,
                    transaction_cost=tc,
                    position_size=position_size,
                    risk_mode=risk_mode,
                    use_limit=use_limit,
                    limit_offset=limit_offset,
                    mr_for_mask=0.0,
                    volatility_col=volatility_col,
                    probability_col=probability_col,
                    tf=tf,
                )
                val_results[model_type] = val_res

                best = sweep_min_return(
                    prices=val_prices,
                    df_pred=val_res["predictions"],
                    ku=ku,
                    kd=kd,
                    hold=hold,
                    min_grid=min_returns,
                    artifacts_dir=DATA_DIR,
                    slippage=0.0,
                    transaction_cost=tc,
                    compound=True,
                    use_limit=use_limit,
                    limit_offset=limit_offset,
                    min_trades=min_trades,
                    volatility_col=volatility_col,
                    probability_col=prob_col_name,
                    tf=tf,
                    risk_mode=risk_mode,
                    position_size=position_size,
                )
                best_per_model[model_type] = best

                mrs_this = [
                    best["best_return"]["mr"],
                    best["best_martin"]["mr"],
                    *base_mrs,
                ]
                mrs_unique = [mr for mr in dict.fromkeys(mrs_this) if mr is not None]
                best_mrs_per_model[model_type] = mrs_unique

            # TEST: final evaluation
            test_results = {}
            for model_type in model_types:
                test_mrs = best_mrs_per_model[model_type]
                test_res = load_predict_DL(
                    df_test,
                    test_prices,
                    ku,
                    kd,
                    hold,
                    window_size,
                    base_name,
                    base_dir=DATA_DIR,
                    dropout=dropout,
                    hidden_size=hidden_size,
                    num_layers=num_layers,
                    target=target,
                    model_type=model_type,
                    min_return=test_mrs,
                    slippage=0.0,
                    transaction_cost=tc,
                    position_size=position_size,
                    risk_mode=risk_mode,
                    use_limit=use_limit,
                    limit_offset=limit_offset,
                    mr_for_mask=0.0,
                    volatility_col=volatility_col,
                    probability_col=probability_col,
                    tf=tf,
                )
                test_results[model_type] = test_res

            key = (tf, metric_name)
            all_results[key] = {
                "tf": tf,
                "es_metric": metric_name,
                "val_bnh": val_bnh,
                "test_bnh": test_bnh,
                "val_results": val_results,
                "test_results": test_results,
                "best_per_model": best_per_model,
                "best_mrs_per_model": best_mrs_per_model,
                "baseline_val": baseline_val,
                "baseline_test": baseline_test,
                "baseline_best": baseline_best,
            }

    return all_results

In [42]:
def hpo_gru_brier_1h(df_1h: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    # 1) single split reused for all HPs (no data leakage across configs)
    df_train, df_val, df_test, scaler = data_pipe(
        df_1h, ku, kd, hold, window_size, volatility_col=volatility_col
    )

    val_prices = (
        df_val[["high", "low", "close", "y", volatility_col]]
        .iloc[window_size - 1 :]
        .reset_index(drop=True)
    )
    test_prices = (
        df_test[["high", "low", "close", "y", volatility_col]]
        .iloc[window_size - 1 :]
        .reset_index(drop=True)
    )

    hpo_rows = []
    best_score = -np.inf
    best_cfg = None

    for hidden_size, dropout, num_layers in itertools.product(
        hidden_grid, dropout_grid, num_layers_grid
    ):
        print("=" * 80)
        print(
            f"HPO trial: hidden={hidden_size}, dropout={dropout}, "
            f"layers={num_layers}"
        )

        # 2) train GRU with Brier ES on this split
        _ = train_DL_panel(
            df_train,
            df_val,
            ku,
            kd,
            hold,
            window_size,
            lr,
            base_name,
            base_dir=DATA_DIR,
            target=target,
            model_types=[model_type],
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            volatility_col=volatility_col,
            probability_col=probability_col,
            es_metric=es_metrics[metric_name],
        )

        # 3) VAL: predictions + min_return sweep
        val_res = load_predict_DL(
            df_val,
            val_prices,
            ku,
            kd,
            hold,
            window_size,
            base_name,
            base_dir=DATA_DIR,
            dropout=dropout,
            hidden_size=hidden_size,
            num_layers=num_layers,
            target=target,
            model_type=model_type,
            min_return=base_mrs,         # for base trade metrics; sweep uses full grid
            slippage=0.0,
            transaction_cost=tc,
            position_size=position_size,
            risk_mode=risk_mode,
            use_limit=use_limit,
            limit_offset=limit_offset,
            mr_for_mask=0.0,
            volatility_col=volatility_col,
            probability_col=probability_col,
            tf=tf,
        )

        # sweep over full min_returns grid on VAL
        best = sweep_min_return(
            prices=val_prices,
            df_pred=val_res["predictions"],
            ku=ku,
            kd=kd,
            hold=hold,
            min_grid=min_returns,
            artifacts_dir=DATA_DIR,
            slippage=0.0,
            transaction_cost=tc,
            compound=True,
            use_limit=use_limit,
            limit_offset=limit_offset,
            min_trades=min_trades,
            volatility_col=volatility_col,
            probability_col=probability_col,
            tf=tf,
            risk_mode=risk_mode,
            position_size=position_size,
        )

        best_ret = best["best_return"]
        best_mrt = best["best_martin"]

        # if no mr satisfies min_trades, treat as very bad config
        if best_mrt["mr"] is None or best_mrt["metrics"] is None:
            val_martin = -np.inf
            val_total_ret = -np.inf
            val_mr_star = None
            val_num_trades = 0
        else:
            val_martin = best_mrt["metrics"]["martin_ratio"]
            val_total_ret = best_mrt["metrics"]["total_return"]
            val_mr_star = best_mrt["mr"]
            val_num_trades = best_mrt["metrics"].get("num_trades", 0)

        # classification metrics (unmasked) on VAL
        mm_raw_val = val_res.get("model_metrics_raw", {})
        val_bss = float(mm_raw_val.get("bss", np.nan))
        val_macro_f1 = float(mm_raw_val.get("macro_f1", np.nan))

        # 4) TEST: evaluate at the VAL-best Martin mr (for diagnostics only, not selection)
        test_martin = np.nan
        test_total_ret = np.nan
        test_sharpe = np.nan
        if val_mr_star is not None and np.isfinite(val_martin):
            test_res = load_predict_DL(
                df_test,
                test_prices,
                ku,
                kd,
                hold,
                window_size,
                base_name,
                base_dir=DATA_DIR,
                dropout=dropout,
                hidden_size=hidden_size,
                num_layers=num_layers,
                target=target,
                model_type=model_type,
                min_return=[val_mr_star],
                slippage=0.0,
                transaction_cost=tc,
                position_size=position_size,
                risk_mode=risk_mode,
                use_limit=use_limit,
                limit_offset=limit_offset,
                mr_for_mask=0.0,
                volatility_col=volatility_col,
                probability_col=probability_col,
                tf=tf,
            )
            tm_test = test_res[val_mr_star]["trade_metrics"]
            test_martin = tm_test.get("martin_ratio", np.nan)
            test_total_ret = tm_test.get("total_return", np.nan)
            test_sharpe = tm_test.get("sharpe_ratio", np.nan)

        # 5) record everything
        row = dict(
            tf=tf,
            es_metric=metric_name,
            model_type=model_type,
            hidden_size=hidden_size,
            dropout=dropout,
            num_layers=num_layers,
            window_size=window_size,
            hold=hold,
            ku=ku,
            kd=kd,
            val_mr_star=val_mr_star,
            val_martin=val_martin,
            val_total_return=val_total_ret,
            val_num_trades=val_num_trades,
            val_bss_raw=val_bss,
            val_macro_f1_raw=val_macro_f1,
            test_martin=test_martin,
            test_total_return=test_total_ret,
            test_sharpe=test_sharpe,
        )
        hpo_rows.append(row)

        # 6) HPO selection (VAL only)
        # primary: val_martin
        # tie-break: val_bss_raw, then val_macro_f1_raw, then val_num_trades
        score = val_martin
        if not np.isfinite(score):
            continue  # skip hopeless configs

        def better(a, b):
            """return True if row a is strictly better than row b under our rule"""
            if a["val_martin"] > b["val_martin"] + 1e-8:
                return True
            if abs(a["val_martin"] - b["val_martin"]) <= 1e-8:
                if a["val_bss_raw"] > b["val_bss_raw"] + 1e-8:
                    return True
                if abs(a["val_bss_raw"] - b["val_bss_raw"]) <= 1e-8:
                    if a["val_macro_f1_raw"] > b["val_macro_f1_raw"] + 1e-8:
                        return True
                    if abs(a["val_macro_f1_raw"] - b["val_macro_f1_raw"]) <= 1e-8:
                        if a["val_num_trades"] > b["val_num_trades"]:
                            return True
            return False

        if best_cfg is None:
            best_cfg = row
            best_score = score
        else:
            if better(row, best_cfg):
                best_cfg = row
                best_score = score

        print(
            f"VAL: Martin={val_martin:.3f}, total_ret={val_total_ret:.2f}, "
            f"BSS={val_bss:.4f}, macro_f1={val_macro_f1:.3f}, "
            f"mr*={val_mr_star}, num_trades={val_num_trades}"
        )
        print(
            f"TEST (diag): Martin={test_martin:.3f}, total_ret={test_total_ret:.2f}, "
            f"Sharpe={test_sharpe:.3f}"
        )

    hpo_df = pd.DataFrame(hpo_rows)

    # sort for inspection (VAL-based)
    hpo_df_sorted = hpo_df.sort_values(
        by=["val_martin", "val_bss_raw", "val_macro_f1_raw", "val_num_trades"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    # save results
    hpo_df_sorted.to_csv(f"gru_brier_1h_ws{window_size}_hold{hold}_hpo_results.csv", index=False)

    print("\nBest config (VAL-based):")
    print(best_cfg)

    return hpo_df_sorted, best_cfg

In [43]:
def add_test_model_metrics_to_hpo(
    hpo_df: pd.DataFrame,
    df_1h: pd.DataFrame,
    save_path: str = "gru_brier_1h_hpo_results_with_model_metrics.csv",
) -> pd.DataFrame:
    """
    For each HPO config, load the trained GRU model and compute
    test classification metrics (raw & masked). Adds columns:

      test_macro_f1_raw,    test_brier_raw,    test_bss_raw,
      test_tp_precision_raw,test_tp_recall_raw,test_tp_f1_raw,
      test_macro_f1_masked, test_brier_masked, test_bss_masked,
      test_tp_precision_masked, test_tp_recall_masked, test_tp_f1_masked

    and saves to `save_path`.
    """
    # 1) Recreate test split once
    df_train, df_val, df_test, scaler = data_pipe(
        df_1h, ku, kd, hold, window_size, volatility_col=volatility_col
    )

    test_prices = (
        df_test[["high", "low", "close", "y", volatility_col]]
        .iloc[window_size - 1 :]
        .reset_index(drop=True)
    )

    # Ensure we work on a copy
    hpo_df = hpo_df.copy()

    # 2) Loop over all HPO rows and compute test model metrics
    for idx, row in hpo_df.iterrows():
        hidden_size = int(row["hidden_size"])
        dropout = float(row["dropout"])
        num_layers = int(row["num_layers"])

        print(
            f"[TEST METRICS] idx={idx}, hidden={hidden_size}, "
            f"dropout={dropout}, layers={num_layers}"
        )

        # We only need model metrics; min_return can be anything (e.g. [0.0])
        test_res = load_predict_DL(
            df_test,
            test_prices,
            ku,
            kd,
            hold,
            window_size,
            base_name,              # same base_name as in HPO
            base_dir=DATA_DIR,
            dropout=dropout,
            hidden_size=hidden_size,
            num_layers=num_layers,
            target=target,
            model_type="gru",       # we did HPO on GRU
            min_return=[0.0],
            slippage=0.0,
            transaction_cost=tc,
            position_size=position_size,
            risk_mode=risk_mode,
            use_limit=use_limit,
            limit_offset=limit_offset,
            mr_for_mask=0.0,
            volatility_col=volatility_col,
            probability_col=probability_col,
            tf="1h",
        )

        mm_raw = test_res.get("model_metrics_raw", {})
        mm_mask = test_res.get("model_metrics_masked", {})

        # Safely extract metrics if present in your triple_barrier_metrics dict
        def get(m, key):
            return float(m.get(key, np.nan))

        # RAW
        hpo_df.loc[idx, "test_macro_f1_raw"]     = get(mm_raw, "macro_f1")
        hpo_df.loc[idx, "test_brier_raw"]        = get(mm_raw, "brier_score")
        hpo_df.loc[idx, "test_bss_raw"]          = get(mm_raw, "bss")
        hpo_df.loc[idx, "test_tp_precision_raw"] = get(mm_raw, "tp_precision")
        hpo_df.loc[idx, "test_tp_recall_raw"]    = get(mm_raw, "tp_recall")
        hpo_df.loc[idx, "test_tp_f1_raw"]        = get(mm_raw, "tp_f1")

        # MASKED (decision bars)
        hpo_df.loc[idx, "test_macro_f1_masked"]     = get(mm_mask, "macro_f1")
        hpo_df.loc[idx, "test_brier_masked"]        = get(mm_mask, "brier_score")
        hpo_df.loc[idx, "test_bss_masked"]          = get(mm_mask, "bss")
        hpo_df.loc[idx, "test_tp_precision_masked"] = get(mm_mask, "tp_precision")
        hpo_df.loc[idx, "test_tp_recall_masked"]    = get(mm_mask, "tp_recall")
        hpo_df.loc[idx, "test_tp_f1_masked"]        = get(mm_mask, "tp_f1")

    # 3) Save and return
    hpo_df.to_csv(save_path, index=False)
    print(f"[OK] Updated HPO results with test model metrics saved to {save_path}")
    return hpo_df

In [44]:
# ku/kd sweep

# ---------------------------------
# HELPER
# ---------------------------------
def _get_metric(d, key):
    return float(d.get(key, np.nan)) if isinstance(d, dict) else np.nan

# ---------------------------------
# MAIN SWEEP FUNCTION
# ---------------------------------
def ku_kd_sweep_gru_configs(
    df_1h: pd.DataFrame,
    save_path: str = "gru_brier_1h_ku_kd_sweep.csv",
    base_name_ku_sweep = "btc_sweep"
) -> pd.DataFrame:
    """
    For each (ku,kd) and each selected GRU config:
      - build labels via data_pipe
      - train GRU (Brier ES)
      - sweep min_return on VAL (best_return & best_martin)
      - evaluate both selected mrs (and base_mrs) on TEST
      - collect trade + classification metrics
    """

    rows = []

    for ku_val, kd_val in KU_KD_GRID:
        print("\n" + "#" * 80)
        print(f"=== ku={ku_val}, kd={kd_val} ===")

        # 1) Rebuild split for this ku/kd
        df_train, df_val, df_test, scaler = data_pipe(
            df_1h, ku_val, kd_val, hold, window_size, volatility_col=volatility_col
        )

        val_prices = (
            df_val[["high", "low", "close", "y", volatility_col]]
            .iloc[window_size - 1 :]
            .reset_index(drop=True)
        )
        test_prices = (
            df_test[["high", "low", "close", "y", volatility_col]]
            .iloc[window_size - 1 :]
            .reset_index(drop=True)
        )

        for cfg in GRU_CONFIGS:
            hidden_size = cfg["hidden_size"]
            dropout = cfg["dropout"]
            num_layers = cfg["num_layers"]
            name = cfg["name"]

            print("-" * 80)
            print(f"[TRAIN] {name} | ku={ku_val}, kd={kd_val}")

            # 2) Train GRU with Brier ES on this split
            _ = train_DL_panel(
                df_train,
                df_val,
                ku_val,
                kd_val,
                hold,
                window_size,
                lr,
                base_name_ku_sweep,
                base_dir=DATA_DIR,
                target=target,
                model_types=[model_type],
                hidden_size=hidden_size,
                num_layers=num_layers,
                dropout=dropout,
                volatility_col=volatility_col,
                probability_col=probability_col,
                es_metric=es_metrics[metric_name],
            )

            # 3) VAL: predictions + base-mr run
            val_res = load_predict_DL(
                df_val,
                val_prices,
                ku_val,
                kd_val,
                hold,
                window_size,
                base_name_ku_sweep,
                base_dir=DATA_DIR,
                dropout=dropout,
                hidden_size=hidden_size,
                num_layers=num_layers,
                target=target,
                model_type=model_type,
                min_return=base_mrs,
                slippage=0.0,
                transaction_cost=tc,
                position_size=position_size,
                risk_mode=risk_mode,
                use_limit=use_limit,
                limit_offset=limit_offset,
                mr_for_mask=0.0,
                volatility_col=volatility_col,
                probability_col=probability_col,
                tf=tf,
            )

            df_pred_val = val_res["predictions"]

            # classification metrics (raw) on VAL
            mm_raw_val = val_res.get("model_metrics_raw", {})
            val_macro_f1 = _get_metric(mm_raw_val, "macro_f1")
            val_brier = _get_metric(mm_raw_val, "brier_score")
            val_bss = _get_metric(mm_raw_val, "bss")
            val_tp_prec = _get_metric(mm_raw_val, "tp_precision")
            val_tp_rec = _get_metric(mm_raw_val, "tp_recall")
            val_tp_f1 = _get_metric(mm_raw_val, "tp_f1")

            # 4) sweep min_return on VAL for this ku/kd + model
            best = sweep_min_return(
                prices=val_prices,
                df_pred=df_pred_val,
                ku=ku_val,
                kd=kd_val,
                hold=hold,
                min_grid=min_returns,
                artifacts_dir=DATA_DIR,
                slippage=0.0,
                transaction_cost=tc,
                compound=True,
                use_limit=use_limit,
                limit_offset=limit_offset,
                min_trades=min_trades,
                volatility_col=volatility_col,
                probability_col=probability_col,
                tf=tf,
                risk_mode=risk_mode,
                position_size=position_size,
            )

            best_ret = best["best_return"]
            best_mrt = best["best_martin"]

            val_mr_best_return = best_ret["mr"]
            val_mr_best_martin = best_mrt["mr"]

            # build set of mrs to evaluate on TEST
            mrs_to_eval = []
            for mr in [val_mr_best_return, val_mr_best_martin] + base_mrs:
                if mr is not None and mr not in mrs_to_eval:
                    mrs_to_eval.append(mr)

            print(f"[VAL] {name} ku={ku_val},kd={kd_val}: mrs_to_eval={mrs_to_eval}")

            # 5) TEST: evaluate each mr
            if mrs_to_eval:
                test_res = load_predict_DL(
                    df_test,
                    test_prices,
                    ku_val,
                    kd_val,
                    hold,
                    window_size,
                    base_name_ku_sweep,
                    base_dir=DATA_DIR,
                    dropout=dropout,
                    hidden_size=hidden_size,
                    num_layers=num_layers,
                    target=target,
                    model_type=model_type,
                    min_return=mrs_to_eval,
                    slippage=0.0,
                    transaction_cost=tc,
                    position_size=position_size,
                    risk_mode=risk_mode,
                    use_limit=use_limit,
                    limit_offset=limit_offset,
                    mr_for_mask=0.0,
                    volatility_col=volatility_col,
                    probability_col=probability_col,
                    tf=tf,
                )

                mm_raw_test = test_res.get("model_metrics_raw", {})
                test_macro_f1 = _get_metric(mm_raw_test, "macro_f1")
                test_brier = _get_metric(mm_raw_test, "brier_score")
                test_bss = _get_metric(mm_raw_test, "bss")
                test_tp_prec = _get_metric(mm_raw_test, "tp_precision")
                test_tp_rec = _get_metric(mm_raw_test, "tp_recall")
                test_tp_f1 = _get_metric(mm_raw_test, "tp_f1")
            else:
                test_res = None
                test_macro_f1 = test_brier = test_bss = np.nan
                test_tp_prec = test_tp_rec = test_tp_f1 = np.nan

            # 6) collect rows for each mr in mrs_to_eval
            for mr in mrs_to_eval:
                tm_val = None
                tm_test = None

                # get VAL trade metrics if available
                if mr in val_res:
                    tm_val = val_res[mr]["trade_metrics"]

                # get TEST trade metrics
                if test_res is not None and mr in test_res:
                    tm_test = test_res[mr]["trade_metrics"]

                row = dict(
                    tf=tf,
                    es_metric=metric_name,
                    model_type=model_type,
                    model_name=name,
                    ku=ku_val,
                    kd=kd_val,
                    hidden_size=hidden_size,
                    dropout=dropout,
                    num_layers=num_layers,
                    window_size=window_size,
                    hold=hold,
                    mr=mr,
                    # selection tag
                    selection=(
                        "best_return" if mr == val_mr_best_return
                        else "best_martin" if mr == val_mr_best_martin
                        else "base"
                    ),
                    # VAL trade metrics
                    val_total_return=_get_metric(tm_val or {}, "total_return"),
                    val_martin=_get_metric(tm_val or {}, "martin_ratio"),
                    val_sharpe=_get_metric(tm_val or {}, "sharpe_ratio"),
                    val_num_trades=_get_metric(tm_val or {}, "num_trades"),
                    # TEST trade metrics
                    test_total_return=_get_metric(tm_test or {}, "total_return"),
                    test_martin=_get_metric(tm_test or {}, "martin_ratio"),
                    test_sharpe=_get_metric(tm_test or {}, "sharpe_ratio"),
                    test_num_trades=_get_metric(tm_test or {}, "num_trades"),
                    # VAL classification (raw)
                    val_macro_f1_raw=val_macro_f1,
                    val_brier_raw=val_brier,
                    val_bss_raw=val_bss,
                    val_tp_precision_raw=val_tp_prec,
                    val_tp_recall_raw=val_tp_rec,
                    val_tp_f1_raw=val_tp_f1,
                    # TEST classification (raw)
                    test_macro_f1_raw=test_macro_f1,
                    test_brier_raw=test_brier,
                    test_bss_raw=test_bss,
                    test_tp_precision_raw=test_tp_prec,
                    test_tp_recall_raw=test_tp_rec,
                    test_tp_f1_raw=test_tp_f1,
                )
                rows.append(row)

    sweep_df = pd.DataFrame(rows)
    sweep_df.to_csv(save_path, index=False)
    print(f"[OK] ku/kd sweep saved to {save_path}")
    return sweep_df


# Example call:
# sweep_results = ku_kd_sweep_gru_configs(df_1h)


In [45]:
def run_block_ablation_experiment(
    df: pd.DataFrame,
    *,
    ku: float,
    kd: float,
    hold: int,
    window_size: int,
    base_name: str,
    DATA_DIR: str,
    model_types: list[str],
    volatility_col: str,
    probability_col: str,
    lr: float,
    hidden_size: int,
    num_layers: int,
    dropout: float,
    tc: float,
    use_limit: bool,
    limit_offset: float,
    mr_fixed: float,
    tf: str = '1h',
    es_metric = None,
) -> pd.DataFrame:
    """
    For each block in FEATURE_BLOCKS:
      - drop that block from features (keep all others),
      - train models from scratch on reduced features,
      - evaluate on val & test at fixed mr_fixed,
      - collect metrics into a single DataFrame.

    Only reduced-feature models are trained; full-feature model is assumed
    to already exist (you can merge with its metrics later).
    """

    non_feature_cols = [
        "timestamp", "open", "high", "low", "close",
        volatility_col, "y"
    ]
    non_feature_cols = [c for c in non_feature_cols if c in df.columns]

    all_ablation_rows = []

    # shared BnH will be re-computed per ablation (same prices)
    for block_name, block_feats in FEATURE_BLOCKS.items():
        print("\n" + "#" * 80)
        print(f"ABLATION: dropping block '{block_name}'")

        # features to KEEP = all features minus this block
        keep_features = [f for f in ALL_FEATURES if f not in set(block_feats)]
        keep_features = [f for f in keep_features if f in df.columns]

        print(f"  Keeping {len(keep_features)} features, "
              f"dropping {len(block_feats)} from block '{block_name}'")

        df_sub = df[non_feature_cols + keep_features].copy()

        # --- split & scale
        df_train, df_val, df_test, scaler = data_pipe(
            df_sub,
            ku,
            kd,
            hold,
            window_size,
            volatility_col=volatility_col,
        )

        # --- prices (as in your DL pipeline)
        val_prices = df_val[["high", "low", "close", "y", volatility_col]].iloc[window_size-1:].reset_index(drop=True)
        test_prices = df_test[["high", "low", "close", "y", volatility_col]].iloc[window_size-1:].reset_index(drop=True)

        val_bnh  = (val_prices["close"].iloc[-1]  / val_prices["close"].iloc[0]  - 1) * 100
        test_bnh = (test_prices["close"].iloc[-1] / test_prices["close"].iloc[0] - 1) * 100

        # --- train on reduced features
        exp_base = f"{base_name}_minus_{block_name}"

        _ = train_DL_panel(
            df_train,
            df_val,
            ku=ku,
            kd=kd,
            hold=hold,
            window_size=window_size,
            lr=lr,
            base_name=exp_base,
            base_dir=DATA_DIR,
            target=["y"],
            model_types=model_types,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            probability_col=probability_col,
            volatility_col=volatility_col,
            es_metric=es_metric,
            
        )

        # --- eval on val & test at fixed mr ---
        base_mrs = [mr_fixed]
        val_results = {}
        test_results = {}

        for model_type in model_types:
            # VAL
            val_res = load_predict_DL(
                df_val,
                val_prices,
                ku,
                kd,
                hold,
                window_size,
                base_name=exp_base,
                base_dir=DATA_DIR,
                dropout=dropout,
                hidden_size=hidden_size,
                num_layers=num_layers,
                target=["y"],
                model_type=model_type,
                min_return=base_mrs,
                slippage=0.0,
                transaction_cost=tc,
                position_size=0.01,
                risk_mode="fixed_risk",
                use_limit=use_limit,
                limit_offset=limit_offset,
                mr_for_mask=0.0,
                volatility_col=volatility_col,
                probability_col=probability_col,
                tf=tf
            )
            val_results[model_type] = val_res

            # TEST
            test_res = load_predict_DL(
                df_test,
                test_prices,
                ku,
                kd,
                hold,
                window_size,
                base_name=exp_base,
                base_dir=DATA_DIR,
                dropout=dropout,
                hidden_size=hidden_size,
                num_layers=num_layers,
                target=["y"],
                model_type=model_type,
                min_return=[mr_fixed],
                slippage=0.0,
                transaction_cost=tc,
                position_size=0.01,
                risk_mode="fixed_risk",
                use_limit=use_limit,
                limit_offset=limit_offset,
                mr_for_mask=0.0,
                volatility_col=volatility_col,
                probability_col=probability_col,
                tf=tf
            )
            test_results[model_type] = test_res

        # --- gather metrics for this ablation ---
        for model_type in model_types:
            vr = val_results[model_type]
            tr = test_results[model_type]

            vm_raw = vr["model_metrics_raw"]
            tm_raw = tr["model_metrics_raw"]

            vtrade = vr[mr_fixed]["trade_metrics"]
            ttrade = tr[mr_fixed]["trade_metrics"]

            row = {
                "dropped_block": block_name,
                "model": model_type,
                "n_features_used": len(keep_features),

                # ---- val (unmasked) ----
                "val_tp_precision": vm_raw.get("tp_precision", np.nan),
                "val_tp_recall": vm_raw.get("tp_recall", np.nan),
                "val_tp_f1": vm_raw.get("tp_f1", np.nan),
                "val_macro_f1": vm_raw.get("macro_f1", np.nan),
                "val_brier": vm_raw.get("brier", np.nan),
                "val_bss": vm_raw.get("bss", np.nan),

                "val_total_return": vtrade.get("total_return", np.nan),
                "val_martin": vtrade.get("martin_ratio", np.nan),
                "val_calmar": vtrade.get("calmar_ratio", np.nan),
                "val_sharpe": vtrade.get("sharpe_ratio", np.nan),
                "val_num_trades": vtrade.get("num_trades", np.nan),

                # ---- test (unmasked) ----
                "test_tp_precision": tm_raw.get("tp_precision", np.nan),
                "test_tp_recall": tm_raw.get("tp_recall", np.nan),
                "test_tp_f1": tm_raw.get("tp_f1", np.nan),
                "test_macro_f1": tm_raw.get("macro_f1", np.nan),
                "test_brier": tm_raw.get("brier", np.nan),
                "test_bss": tm_raw.get("bss", np.nan),

                "test_total_return": ttrade.get("total_return", np.nan),
                "test_martin": ttrade.get("martin_ratio", np.nan),
                "test_calmar": ttrade.get("calmar_ratio", np.nan),
                "test_sharpe": ttrade.get("sharpe_ratio", np.nan),
                "test_num_trades": ttrade.get("num_trades", np.nan),

                "val_bnh": val_bnh,
                "test_bnh": test_bnh,
                "mr_fixed": mr_fixed,
            }
            all_ablation_rows.append(row)

    ablation_df = pd.DataFrame(all_ablation_rows)
    ablation_df = ablation_df.sort_values(
        ["model", "dropped_block"]
    ).reset_index(drop=True)

    return ablation_df

## Main

In [46]:
if in_kaggle():
    DATA_DIR = "/kaggle/working"
elif in_colab():
    DATA_DIR = "/content"   # typical Colab working dir
else:
    DATA_DIR = "artifacts"

In [47]:
# PARAMS

target = ['y']
# X_cols = CORE_PRICE_VOL
volatility_col = 'atr_200'
probability_col = 'p1'

window_grid=[48, 96]

lr = 1e-4

num_classes = 2
dropout = 0.1
dropout_grid = [0.1, 0.5]

num_layers_grid=[2]
num_layers = 1

hidden_grid=[128, 384, 512]
hidden_size = 384

ku = 6
kd = 2

min_returns = np.linspace(0, 0.5, int(200+1))

use_limit = True
limit_offset = 0.0
position_size = 0.05
risk_mode = 'fixed_risk'
min_return = 0.2
tc = True

tf = '1h'
if tf == '1h':
    min_trades = 115
elif tf == '15m': 
    min_trades = 230
else:
    min_trades = 0

if tf == '1h':
    hold = 336
    window_size = 336
elif tf == '4h':
    hold = 84
    window_size = 84

batch_size = 256
metric = 'brier'
# base_name = f'rf-vtc_{tf}_new-onchain_{metric}'
base_name = f'btc-new_{tf}_{metric}'

model_types = ['lstm', 'bilstm', 'gru', 'bigru','gru_cat']

In [ ]:
# Fetch data
df = fetch_data(tf)

df_1h = fetch_data('1h')

print(list(df.columns))
print(len(list(df.columns)))
print(len(df))
# df

['open', 'high', 'low', 'close', 'volume', 'funding_rate', 'ema_20', 'ema_50', 'ema_200', 'adx', 'rsi_14', 'bb_percent', 'bb_width', 'atr_200', 'atr_vol_regime', 'mfi_14', 'z_volume', 'vwap', 'vwap_distance', 'log_return_1h', 'log_return_4h', 'log_return_1d', 'roll_std_4h', 'roll_std_8h', 'funding_bias', 'vwap_session', 'session_vol_mean', 'session_return_mean', 'session_volatility', 'fd_24', 'fd_slope', 'fd_ema_12', 'fd_ema_24', 'fd_trend_strength', 'fd_threshold_causal', 'fd_regime', 'fd_regime_switch', 'fd_volatility', 'fd_vol_ratio', 'fd_vol_slope', 'fd_slope_atr_norm', 'fd_entropy', 'fd_vol_adjusted', 'fd_24_robust_z', 'fd_ema_12_robust_z', 'fd_ema_24_robust_z', 'fd_trend_strength_robust_z', 'fd_slope_robust_z', 'fd_slope_atr_norm_robust_z', 'fd_volatility_robust_z', 'fd_vol_ratio_robust_z', 'fd_vol_slope_robust_z', 'fd_entropy_robust_z', 'fd_vol_adjusted_robust_z', 'pattern_bullish_engulf', 'pattern_bearish_engulf', 'pattern_harami', 'pattern_hammer', 'pattern_inverted_hammer', '

In [12]:
#training
df_train, df_val, df_test, scaler = data_pipe(df, ku, kd, hold, window_size, volatility_col=volatility_col)

output = train_DL_panel(
                            df_train,
                            df_val,
                            ku, 
                            kd, 
                            hold, 
                            window_size, 
                            lr, 
                            base_name, 
                            base_dir = DATA_DIR, 
                            target = target, 
                            model_types=model_types, 
                            #    min_return = 1.0,
                            hidden_size = hidden_size,
                            num_layers = num_layers,
                            dropout=dropout,
                            volatility_col=volatility_col,
                            probability_col=probability_col,
                            es_metric=es_metrics[metric]
                  )


['open', 'high', 'low', 'close', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'z_volume', 'bb_percent', 'atr_200']
[INFO] Using StandardScaler, scaling = True
[INFO] Total features: 109 | Scaled: 70 | Excluded: 11
[OK] Standard scaling applied to 70 columns (fit on train only).
[OK] Split complete → Train: 43521, Val: 4777, Test: 4777
=== 3-class label distribution per split (before binarization) ===
split   train           val          test        
metric  count    prop count    prop count    prop
0          32  0.0007     0  0.0000     0  0.0000
1       32259  0.7412  3427  0.7174  3288  0.6883
2       11230  0.2580  1350  0.2826  1489  0.3117
y
0    0.741964
1    0.258036
Name: proportion, dtype: float64
[0.67388743 1.9377115 ]
Training  lstm


Epoch 1/50:   0%|          | 0/169 [00:00<?, ?it/s]

Epoch 2/50:   0%|          | 0/169 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/169 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/169 [00:00<?, ?it/s]

...

GRU pretrain 6/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 7/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 8/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 9/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 10/50:   0%|          | 0/169 [00:00<?, ?it/s]

In [ ]:
#TRAIN METRICS
for model in model_types:
    print(model) 
    print("Train metrics: ", output[model]['train_metrics'])
    print("Val metrics: ", output[model]['val_metrics'])

In [ ]:
#PERFORMANCE TEST 

# --- Split ---
df_train, df_val, df_test, scaler = data_pipe(
    df, ku, kd, hold, window_size, volatility_col=volatility_col
)
df_test = df_test.iloc[int(len(df_test)/2):]

val_prices = df_val[["high", "low", "close", "y", volatility_col]].iloc[window_size-1:].reset_index(drop=True)
test_prices = df_test[["high", "low", "close", "y", volatility_col]].iloc[window_size-1:].reset_index(drop=True)
test_prices = test_prices.iloc[int(len(test_prices)/2):]

# --- BnH ---
val_bnh  = (val_prices["close"].iloc[-1]  / val_prices["close"].iloc[0]  - 1) * 100
test_bnh = (test_prices["close"].iloc[-1] / test_prices["close"].iloc[0] - 1) * 100

# Base mrs just for reference trading run
base_mrs = [0.0]

# ---------------- VAL: predictions + base-mr trading ----------------
val_results = {}

for model_type in model_types:
    val_results[model_type] = load_predict_DL(
        df_val,
        val_prices,
        ku,
        kd,
        hold,
        window_size,
        base_name,
        base_dir=DATA_DIR,
        dropout=dropout,
        hidden_size=hidden_size,
        num_layers=num_layers,
        target=["y"],
        model_type=model_type,
        min_return=base_mrs,
        slippage=0.0,
        transaction_cost=tc,
        position_size=position_size,
        risk_mode=risk_mode,
        use_limit=use_limit,
        limit_offset=limit_offset,
        mr_for_mask=0.0,
        volatility_col=volatility_col,
        probability_col=probability_col,
        tf=tf,
    )

# ---------------- VAL: sweep min_return per model ----------------
best_per_model = {}
mrs_for_test = {}

for model_type in model_types:
    df_pred_val = val_results[model_type]["predictions"]

    best = sweep_min_return(
        prices=val_prices,
        df_pred=df_pred_val,
        ku=ku,
        kd=kd,
        hold=hold,
        min_grid=min_returns,
        artifacts_dir=DATA_DIR,
        slippage=0.0,
        transaction_cost=tc,
        compound=True,
        use_limit=use_limit,
        limit_offset=limit_offset,
        min_trades=min_trades,
        volatility_col=volatility_col,
        probability_col=probability_col,
        position_size=position_size,
        risk_mode=risk_mode,
        tf=tf,
    )
    best_per_model[model_type] = best

    # Build mr list for test: best_return, best_martin, plus base_mrs
    mrs_this = [
        best["best_return"]["mr"],
        best["best_martin"]["mr"],
    ] + base_mrs
    mrs_this = [mr for mr in mrs_this if mr is not None]

    # unique, keep order
    seen = set()
    mrs_unique = []
    for mr in mrs_this:
        if mr not in seen:
            seen.add(mr)
            mrs_unique.append(mr)

    mrs_for_test[model_type] = mrs_unique

# ---------------- TEST: final evaluation ----------------
test_results = {}
eq = {}

for model_type in model_types:
    test_mrs = mrs_for_test[model_type]

    test_res = load_predict_DL(
        df_test,
        test_prices,
        ku,
        kd,
        hold,
        window_size,
        base_name,
        base_dir=DATA_DIR,
        dropout=dropout,
        hidden_size=hidden_size,
        num_layers=num_layers,
        target=["y"],
        model_type=model_type,
        min_return=test_mrs,
        slippage=0.0,
        transaction_cost=tc,
        position_size=position_size,
        risk_mode=risk_mode,
        use_limit=use_limit,
        limit_offset=limit_offset,
        mr_for_mask=0.0,
        volatility_col=volatility_col,
        probability_col=probability_col,
        tf=tf,
    )
    test_results[model_type] = test_res
    eq[model_type] = test_res[0]['equity_curve']['equity']

# ---------------- CLASSIFICATION DF (UNMASKED ONLY) ----------------
cls_rows = []

def _get(d, key):
    return d.get(key, np.nan)

for model_type in model_types:
    # VAL
    cls_val = val_results[model_type]["model_metrics_raw"]
    cls_rows.append(
        {
            "split": "val",
            "model": model_type,
            "tp_precision": _get(cls_val, "tp_precision"),
            "tp_recall": _get(cls_val, "tp_recall"),
            "tp_f1": _get(cls_val, "tp_f1"),
            "macro_precision": _get(cls_val, "macro_precision"),
            "macro_recall": _get(cls_val, "macro_recall"),
            "macro_f1": _get(cls_val, "macro_f1"),
            "brier": _get(cls_val, "brier"),
            "bss": _get(cls_val, "bss"),
        }
    )

    # TEST
    cls_test = test_results[model_type]["model_metrics_raw"]
    cls_rows.append(
        {
            "split": "test",
            "model": model_type,
            "tp_precision": _get(cls_test, "tp_precision"),
            "tp_recall": _get(cls_test, "tp_recall"),
            "tp_f1": _get(cls_test, "tp_f1"),
            "macro_precision": _get(cls_test, "macro_precision"),
            "macro_recall": _get(cls_test, "macro_recall"),
            "macro_f1": _get(cls_test, "macro_f1"),
            "brier": _get(cls_test, "brier"),
            "bss": _get(cls_test, "bss"),
        }
    )

cls_summary = pd.DataFrame(cls_rows).sort_values(["split", "model"]).reset_index(drop=True)

# ---------------- TRADING DF (WITH CRITERION) ----------------
trade_rows = []

# VAL: only base mrs, criterion="base"
for model_type in model_types:
    for mr in base_mrs:
        tm = val_results[model_type][mr]["trade_metrics"]
        trade_rows.append(
            {
                "split": "val",
                "model": model_type,
                "mr": mr,
                "criterion": "base",
                "sharpe_ratio": _get(tm, "sharpe_ratio"),
                "calmar_ratio": _get(tm, "calmar_ratio"),
                "martin_ratio": _get(tm, "martin_ratio"),
                "total_return": _get(tm, "total_return"),
                "max_drawdown": _get(tm, "max_drawdown"),
                "volatility": _get(tm, "volatility"),
                "winrate": _get(tm, "winrate"),
                "num_trades": _get(tm, "num_trades"),
                "final_equity": _get(tm, "final_equity"),
            }
        )

# TEST: mrs from best_return / best_martin / base_mrs
for model_type in model_types:
    best_ret_mr = best_per_model[model_type]["best_return"]["mr"]
    best_mrt_mr = best_per_model[model_type]["best_martin"]["mr"]

    # map mr -> list of tags
    crit_map = {}
    if best_ret_mr is not None:
        crit_map.setdefault(best_ret_mr, []).append("best_return")
    if best_mrt_mr is not None:
        crit_map.setdefault(best_mrt_mr, []).append("best_martin")
    for mr_base in base_mrs:
        if mr_base not in crit_map:
            crit_map.setdefault(mr_base, []).append("base")

    for mr in mrs_for_test[model_type]:
        tags = crit_map.get(mr, ["base"])
        # precedence: best_return > best_martin > base
        if "best_return" in tags:
            criterion = "best_return"
        elif "best_martin" in tags:
            criterion = "best_martin"
        else:
            criterion = "base"

        tm = test_results[model_type][mr]["trade_metrics"]
        trade_rows.append(
            {
                "split": "test",
                "model": model_type,
                "mr": mr,
                "criterion": criterion,
                "sharpe_ratio": _get(tm, "sharpe_ratio"),
                "calmar_ratio": _get(tm, "calmar_ratio"),
                "martin_ratio": _get(tm, "martin_ratio"),
                "total_return": _get(tm, "total_return"),
                "max_drawdown": _get(tm, "max_drawdown"),
                "volatility": _get(tm, "volatility"),
                "winrate": _get(tm, "winrate"),
                "num_trades": _get(tm, "num_trades"),
                "final_equity": _get(tm, "final_equity"),
            }
        )

# BnH rows in trading DF
trade_rows.append(
    {
        "split": "val",
        "model": "bnh",
        "mr": np.nan,
        "criterion": "bnh",
        "sharpe_ratio": np.nan,
        "calmar_ratio": np.nan,
        "martin_ratio": np.nan,
        "total_return": float(val_bnh),
        "max_drawdown": np.nan,
        "volatility": np.nan,
        "winrate": np.nan,
        "num_trades": np.nan,
        "final_equity": np.nan,
    }
)
trade_rows.append(
    {
        "split": "test",
        "model": "bnh",
        "mr": np.nan,
        "criterion": "bnh",
        "sharpe_ratio": np.nan,
        "calmar_ratio": np.nan,
        "martin_ratio": np.nan,
        "total_return": float(test_bnh),
        "max_drawdown": np.nan,
        "volatility": np.nan,
        "winrate": np.nan,
        "num_trades": np.nan,
        "final_equity": np.nan,
    }
)

trade_summary = pd.DataFrame(trade_rows).sort_values(
    ["split", "model", "criterion", "mr"]
).reset_index(drop=True)

# optional: save
RESULTS_DIR = os.path.join(DATA_DIR, "res")
os.makedirs(RESULTS_DIR, exist_ok=True)

cls_summary.to_csv("res/cls_summary.csv", index=False)
trade_summary.to_csv("res/trade_summary.csv", index=False)




In [ ]:
cls_summary

In [ ]:
trade_summary[trade_summary['mr']==0]

In [ ]:
# BnH stats
df_train, df_val, df_test, scaler = data_pipe(
    df, ku, kd, hold, window_size, volatility_col=volatility_col
)
test_prices = df_test[["high", "low", "close", "y", volatility_col]].iloc[window_size-1:].reset_index(drop=True)

test_bnh_stats = buy_and_hold_stats(test_prices, equity=10000.0, price_col="close")
test_bnh_stats

['open', 'high', 'low', 'close', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'z_volume', 'bb_percent', 'atr_200']
[INFO] Using StandardScaler, scaling = True
[INFO] Total features: 109 | Scaled: 70 | Excluded: 11
[OK] Standard scaling applied to 70 columns (fit on train only).
[OK] Split complete → Train: 41986, Val: 4608, Test: 4609
=== 3-class label distribution per split (before binarization) ===
split   train           val          test        
metric  count    prop count    prop count    prop
0          73  0.0017    18  0.0039     1  0.0002
1       30993  0.7382  3451  0.7489  3158  0.6852
2       10920  0.2601  1139  0.2472  1450  0.3146


{'total_return': 118.64693518344471,
 'final_equity': 21864.693518344473,
 'max_drawdown': -0.2511960655525012}

### Ablation test - effects of feature blocks on classification and trading performance

In [ ]:
ALL_FROM_BLOCKS = sorted({f for fs in FEATURE_BLOCKS.values() for f in fs})
missing = sorted(set(ALL_FEATURES) - set(ALL_FROM_BLOCKS))
if missing:
    print("[WARN] Some features are not assigned to blocks:", missing)

model_types = ['gru_cat']

ablation_results = run_block_ablation_experiment(
    df=df,                   # full df with all features + OHLC + y + atr_200 etc.
    ku=ku,
    kd=kd,
    hold=hold,
    window_size=window_size,
    base_name=base_name,     # same base_name you used for full-feature training
    DATA_DIR=DATA_DIR,
    model_types=model_types, # e.g. ["gru", "lstm"]
    volatility_col=volatility_col,
    probability_col=probability_col,
    lr=lr,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    tc=tc,
    use_limit=use_limit,
    limit_offset=limit_offset,
    mr_fixed=min_return,
    tf=tf,
    es_metric=es_metrics[metric],
)

ablation_results.to_csv(f"{DATA_DIR}/ablation.csv")



################################################################################
ABLATION: dropping block 'core_trend_mom'
  Keeping 84 features, dropping 20 from block 'core_trend_mom'
['open', 'high', 'low', 'close', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'z_volume', 'bb_percent', 'atr_200']
[INFO] Using StandardScaler, scaling = True
[INFO] Total features: 89 | Scaled: 51 | Excluded: 11
[OK] Standard scaling applied to 51 columns (fit on train only).
[OK] Split complete → Train: 43521, Val: 4777, Test: 4777
=== 3-class label distribution per split (before binarization) ===
split   train           val          test        
metric  count    prop count    prop count    prop
0          32  0.0007     0  0.0000     0  0.0000
1       32259  0.7412  3427  0.7174  3288  0.6883
2       11230  0.2580  1350  0.2826  1489  0.3117
y
0    0.741964
1    0.258036
Name: proportion, dtype: float64
[0.67388743 1.9377115 ]
Training  gru_cat


GRU pretrain 1/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 2/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 3/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 4/50:   0%|          | 0/169 [00:00<?, ?it/s]

...

GRU pretrain 7/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 8/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 9/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 10/50:   0%|          | 0/169 [00:00<?, ?it/s]

y
0    0.717396
1    0.282604
Name: proportion, dtype: float64
y
0    0.688298
1    0.311702
Name: proportion, dtype: float64


In [ ]:
ablation_results

### Hyperparameter optimization

In [119]:
# ------HPO PARAMS----------
tf = "1h"
metric_name = "brier"          # ES = Brier
model_type = "gru_cat"             # HPO only for GRU

window_size = 336
hold = 336
min_trades = 115
base_mrs = [0.0]

base_name = f"hpo_{model_type}_{window_size}_{hold}_{tf}_{metric_name}"

# HPO search space (adjust if needed)
hidden_grid = [128, 256, 384, 512]
dropout_grid = [0.1]
num_layers_grid = [1, 2, 3]

In [120]:
# ------HPO----------
hpo_results_df, best_config = hpo_gru_brier_1h(df_1h)

['open', 'high', 'low', 'close', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'z_volume', 'bb_percent', 'atr_200']
[INFO] Using StandardScaler, scaling = True
[INFO] Total features: 109 | Scaled: 70 | Excluded: 11
[OK] Standard scaling applied to 70 columns (fit on train only).
[OK] Split complete → Train: 43521, Val: 4777, Test: 4777
=== 3-class label distribution per split (before binarization) ===
split   train           val          test        
metric  count    prop count    prop count    prop
0          32  0.0007     0  0.0000     0  0.0000
1       32259  0.7412  3427  0.7174  3288  0.6883
2       11230  0.2580  1350  0.2826  1489  0.3117
HPO trial: hidden=128, dropout=0.1, layers=1
y
0    0.741964
1    0.258036
Name: proportion, dtype: float64
[0.67388743 1.9377115 ]
Training  gru_cat


GRU pretrain 1/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 2/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 3/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 4/50:   0%|          | 0/169 [00:00<?, ?it/s]

...

GRU pretrain 1/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 2/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 3/50:   0%|          | 0/169 [00:00<?, ?it/s]

GRU pretrain 4/50:   0%|          | 0/169 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
hpo_results_with_metrics = add_test_model_metrics_to_hpo(hpo_results_df, df_1h)
hpo_results_with_metrics

In [ ]:
RESULTS_DIR = os.path.join(DATA_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
fname = 'HPO_res_gru_brier_1h'
full_path = os.path.join(RESULTS_DIR, fname)
hpo_results_df.to_csv(full_path, index=False)


## ML evaluation

In [24]:
# ML ASSESSMENT

def evaluate_test_only_models(
    model_dfs: dict,
    prices: pd.DataFrame,
    *,
    ku: float,
    kd: float,
    hold: int,
    tf: str,
    atr_col: str = "atr_200",
    y_col: str = "y",
    proba_col: str = "p1",
    pred_col: str = "pred",
    base_mrs=(0.0, 0.06, 0.1),
    position_size: float = 0.01,
    risk_mode: str = "fixed_risk",
    use_limit: bool = True,
    limit_offset: float = 0.0,
    tc: bool = True,
    slippage: float = 0.0,
):
    """
    Parameters
    ----------
    model_dfs : dict[str, pd.DataFrame]
        Mapping model_name -> dataframe with at least columns:
        [y_col, pred_col, proba_col].

        Length must match `prices`.

    prices : pd.DataFrame
        Shared price frame (test slice), e.g. like test_prices in your DL pipeline.
        Must contain at least: ["close", atr_col].
        (If you use OHLC triple-barrier, it can also contain ["high","low"] etc.)

    Returns
    -------
    cls_summary : pd.DataFrame
        Classification metrics (test only, unmasked).

    trade_summary : pd.DataFrame
        Trading metrics (test only) with 'criterion' in {'base', 'bnh'}.
    """

    def _get(d, key):
        return d.get(key, np.nan)

    # --- ensure prices is clean and indexed 0..N-1 ---
    prices = prices.reset_index(drop=True)

    # --- Buy & Hold, once for the shared test window ---
    if "close" not in prices.columns:
        raise ValueError("prices must contain 'close' column")

    bnh_ret = (prices["close"].iloc[-1] / prices["close"].iloc[0] - 1) * 100

    cls_rows = []
    trade_rows = []

    # BnH baseline row (shared for all models)
    trade_rows.append(
        {
            "split": "test",
            "model": "bnh",
            "mr": np.nan,
            "criterion": "bnh",
            "sharpe_ratio": np.nan,
            "calmar_ratio": np.nan,
            "martin_ratio": np.nan,
            "total_return": float(bnh_ret),
            "max_drawdown": np.nan,
            "volatility": np.nan,
            "winrate": np.nan,
            "num_trades": np.nan,
            "final_equity": np.nan,
        }
    )

    # --- per-model evaluation ---
    for model_name, df in model_dfs.items():
        df = df.reset_index(drop=True)

        # length alignment check
        if len(df) != len(prices):
            raise ValueError(
                f"Length mismatch for model '{model_name}': "
                f"len(preds)={len(df)}, len(prices)={len(prices)}"
            )

        # ----- prediction frame for metrics & backtest -----
        if y_col not in df.columns:
            raise ValueError(f"'{y_col}' not found in model df for '{model_name}'")
        if pred_col not in df.columns:
            raise ValueError(f"'{pred_col}' not found in model df for '{model_name}'")
        if proba_col not in df.columns:
            raise ValueError(f"'{proba_col}' not found in model df for '{model_name}'")

        preds = pd.DataFrame(
            {
                "true": df[y_col].astype(int).values,
                "pred": df[pred_col].astype(int).values,
                proba_col: df[proba_col].astype(float).values,
            }
        )

        # ----- classification metrics (unmasked) -----
        cls = triple_barrier_metrics(
            y_true=preds["true"],
            y_pred=preds["pred"],
            p_all=preds[[proba_col]],
            ku=ku,
            kd=kd,
        )

        cls_rows.append(
            {
                "split": "test",
                "model": model_name,
                "tp_precision": _get(cls, "tp_precision"),
                "tp_recall": _get(cls, "tp_recall"),
                "tp_f1": _get(cls, "tp_f1"),
                "macro_precision": _get(cls, "macro_precision"),
                "macro_recall": _get(cls, "macro_recall"),
                "macro_f1": _get(cls, "macro_f1"),
                "brier": _get(cls, "brier"),
                "bss": _get(cls, "bss"),
            }
        )

        # ----- trading metrics for base min_return values -----
        for mr in base_mrs:
            tm, tlog, eq = standard_trade_test(
                predictions=preds,
                prices=prices,
                ku=ku,
                kd=kd,
                hold=hold,
                probability_column=proba_col,
                atr_column=atr_col,
                equity=10000.0,
                position_size=position_size,
                risk_mode=risk_mode,
                compound=True,
                transaction_cost=tc,
                slippage=slippage,
                min_return=mr,
                use_limit=use_limit,
                limit_offset=limit_offset,
                tf=tf,
            )

            trade_rows.append(
                {
                    "split": "test",
                    "model": model_name,
                    "mr": mr,
                    "criterion": "base",
                    "sharpe_ratio": _get(tm, "sharpe_ratio"),
                    "calmar_ratio": _get(tm, "calmar_ratio"),
                    "martin_ratio": _get(tm, "martin_ratio"),
                    "total_return": _get(tm, "total_return"),
                    "max_drawdown": _get(tm, "max_drawdown"),
                    "volatility": _get(tm, "volatility"),
                    "winrate": _get(tm, "winrate"),
                    "num_trades": _get(tm, "num_trades"),
                    "final_equity": _get(tm, "final_equity"),
                }
            )

    cls_summary = (
        pd.DataFrame(cls_rows)
        .sort_values(["split", "model"])
        .reset_index(drop=True)
    )

    trade_summary = (
        pd.DataFrame(trade_rows)
        .sort_values(["split", "model", "criterion", "mr"])
        .reset_index(drop=True)
    )

    return cls_summary, trade_summary

In [128]:
RESULTS_DIR = Path('/kaggle/input/ml-preds')

# === 2. Model name patterns to look for in filenames ===
MODEL_PATTERNS = {
    "cat": "cat",
    "lgbm": "lgbm",
    "rf": "rf",
    "stacking_raw": "stacking_raw",
    "stacking_cal": "stacking_cal",   # in case filename has "stack" instead of "stacking"
    "xgb": "xgb",
}

# === 3. Scan folder and load into dfs ===
model_dfs = {}  # e.g. {"cat": df_cat, "lgbm": df_lgbm, ...}

for path in sorted(RESULTS_DIR.glob("**/*")):
    if not path.is_file():
        continue

    suffix = path.suffix.lower()
    if suffix not in {".csv", ".parquet"}:
        continue

    fname = path.name.lower()

    # detect model by substring in filename
    model_name = None
    for pattern, canonical in MODEL_PATTERNS.items():
        if pattern in fname:
            model_name = canonical
            break

    if model_name is None:
        # file doesn't match any known model pattern
        continue

    # load file
    if suffix == ".csv":
        df = pd.read_csv(path)
    else:  # ".parquet"
        df = pd.read_parquet(path)

    # handle duplicates (if more than one file per model)
    if model_name in model_dfs:
        i = 2
        alt_name = f"{model_name}_{i}"
        while alt_name in model_dfs:
            i += 1
            alt_name = f"{model_name}_{i}"
        print(f"Warning: multiple files for model '{model_name}', "
              f"storing additional as '{alt_name}' from {path.name}")
        model_name = alt_name

    model_dfs[model_name] = df

# === 4. (Optional) expose as df_cat, df_lgbm, df_rf, df_stacking, df_xgb ===
for name, df in model_dfs.items():
    globals()[f"df_{name}"] = df

# Quick check:
list(model_dfs.keys())

['cat', 'lgbm', 'rf', 'stacking_cal', 'stacking_raw', 'xgb']

In [129]:
test_prices = df_test[['high','low','close','y',volatility_col]].reset_index(drop=True)

ml_cls_summary, ml_trade_summary = evaluate_test_only_models(
    model_dfs=model_dfs,          # {"cat": df_cat, "lgbm": df_lgbm, ...}
    prices=test_prices,
    ku=6,
    kd=2,
    hold=336,
    tf="1h",
    atr_col="atr_200",
    y_col="y_true",
    proba_col="p_model",
    pred_col="y_pred",
    base_mrs=(0.0, 0.06, 0.1),
)

ml_cls_summary.to_csv(f"{DATA_DIR}/ml_cls_summary.csv")
ml_trade_summary.to_csv(f"{DATA_DIR}/ml_trade_summary.csv")

In [130]:
ml_cls_summary

,split,model,tp_precision,tp_recall,tp_f1,macro_precision,macro_recall,macro_f1,brier,bss
0,test,cat,0.312272,0.172599,0.222318,0.500344,0.500229,0.487022,0.240633,-0.121603
1,test,lgbm,0.726415,0.051713,0.096552,0.712062,0.521446,0.457749,0.219470,-0.022961
2,test,rf,0.000000,0.000000,0.000000,0.344149,0.500000,0.407688,0.240000,-0.118651
3,test,stacking_cal,0.316547,0.236400,0.270665,0.503157,0.502628,0.498917,0.215886,-0.006255
4,test,stacking_raw,0.793103,0.015447,0.030303,0.742171,0.506811,0.423564,0.231792,-0.080393
5,test,xgb,0.370787,0.022163,0.041825,0.530103,0.502565,0.426128,0.221371,-0.031820


In [131]:
ml_trade_summary

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,split,model,mr,criterion,sharpe_ratio,calmar_ratio,martin_ratio,total_return,max_drawdown,volatility,winrate,num_trades,final_equity
0,test,bnh,NaN,bnh,NaN,NaN,NaN,25.747629,NaN,NaN,NaN,NaN,NaN
1,test,cat,0.00,base,1.583845,3.625737,4.803007,25.275036,-0.141127,0.286491,0.312925,147.0,12527.503597
2,test,cat,0.06,base,1.583845,3.625737,4.803007,25.275036,-0.141127,0.286491,0.312925,147.0,12527.503597
3,test,cat,0.10,base,1.583845,3.625737,4.803007,25.275036,-0.141127,0.286491,0.312925,147.0,12527.503597
4,test,lgbm,0.00,base,-0.458652,-0.763099,-5.414909,-5.920296,-0.138746,0.200622,0.253012,83.0,9407.970393
5,test,lgbm,0.06,base,1.002261,1.810057,-2.281115,7.517547,-0.078538,0.142640,0.333333,36.0,10751.754672
6,test,lgbm,0.10,base,-0.519758,-0.522781,-7.957310,-2.833333,-0.098210,0.093146,0.250000,20.0,9716.666714
7,test,rf,0.00,base,1.157292,6.629299,-31.622984,3.562287,-0.010000,0.056838,0.500000,4.0,10356.228713
8,test,rf,0.06,base,0.000000,NaN,-inf,0.000000,0.000000,0.000000,0.000000,0.0,10000.000000
9,test,rf,0.10,base,0.000000,NaN,-inf,0.000000,0.000000,0.000000,0.000000,0.0,10000.000000
